# Física BR · Taller: Unidades, Medidas y Reporte en Ingeniería
### Física BR — Física Mecánica (Fundamentos)

---
*Física interactiva para ingeniería.* Todo resultado en ingeniería es una **medida**: tiene unidades, cifras significativas y debe reportarse con rigor. Este taller sienta las bases de cómo se expresa y comunica una medida, y por qué eso importa profesionalmente.

**Contenido:**
- **Punto 1:** Conversión de unidades
- **Punto 2:** Cifras significativas en cálculos
- **Punto 3:** Interpretación de gráficas lineales
- **Punto 4:** La conversión correcta como asunto de ingeniería (casos reales)
- **Punto 5:** El reporte adecuado en informes de ingeniería

> 📎 **Nota:** para el tratamiento a profundidad de **errores e incertidumbre** —exactitud vs. precisión, tipos de error, estadística de mediciones y propagación— consulta el taller **"Errores e Incertidumbre en la Medición"** de la misma serie. Este taller se enfoca en unidades, conversión y comunicación del resultado.

> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*


---
## Punto 1 — Conversión de unidades (Ing. Civil/Transporte)  ★★  [CL, UFT]
En un proyecto vial se registra la velocidad de diseño de  90,0 km/h  y un área de sección de
250 cm². Para los cálculos en SI se requieren en  m/s  y  m².

**Análisis dimensional:**  multiplicar por factores de conversión iguales a 1 (ej. 1000 m / 1 km).

> 🔧 **Aplicación en ingeniería:** mezclar unidades es una de las causas más frecuentes (y costosas)
> de error en ingeniería. Convertir correctamente al SI antes de calcular es un hábito que evita
> fallas de diseño. El análisis dimensional también sirve para verificar que una fórmula sea coherente.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega una conversión (ej. 90 km/h → m/s) a un asistente: *"Explícame por qué multiplicar por factores iguales a 1 no cambia la cantidad física, solo su representación. ¿Qué le pasaría al número si invierto el factor por error? No me des el resultado, explícame la lógica."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(8,3.5))
    # ejemplo: 72 km/h -> m/s
    ax.text(0.5,2.5,'72 km/h',fontsize=13,color='#2c3e50',ha='center',fontweight='bold',
            bbox=dict(boxstyle='round',fc='#eaf4fb',ec='#2980b9'))
    # factores de conversión
    factores=[('× 1000 m','1 km','#27ae60'),('× 1 h','3600 s','#e67e22')]
    xpos=2.2
    for num,den,col in factores:
        ax.text(xpos,2.9,num,fontsize=10,color=col,ha='center')
        ax.plot([xpos-0.6,xpos+0.6],[2.5,2.5],color='#2c3e50',lw=1)
        ax.text(xpos,2.1,den,fontsize=10,color=col,ha='center')
        xpos+=1.8
    ax.text(xpos+0.3,2.5,'= 20 m/s',fontsize=13,color='#2c3e50',ha='center',fontweight='bold',
            bbox=dict(boxstyle='round',fc='#eafaf1',ec='#27ae60'))
    # mostrar simplificación
    ax.text(4,1.1,'Las unidades se simplifican como factores:\nkm con km, h con h → queda m/s',
            ha='center',fontsize=9,color='#2c3e50',
            bbox=dict(boxstyle='round',fc='#fdf6e3',ec='#d9a441'))
    # tachar unidades que se simplifican
    ax.annotate('km',xy=(0.5,2.5),xytext=(0.5,2.5),fontsize=1,color='red')
    ax.set_xlim(-0.3,7.5); ax.set_ylim(0.5,3.3); ax.axis('off')
    ax.set_title('Análisis dimensional: multiplicar por factores que valen 1')
    plt.tight_layout(); plt.show()
esquema()

def conversion(v_kmh=90.0):
    v_ms=v_kmh*1000/3600
    vs=np.linspace(0,150,100); vms=vs*1000/3600
    plt.figure(figsize=(9,4)); plt.plot(vs,vms,lw=2,color='teal')
    def _tc3(x):
        import math
        if x==0: return '0,00'
        d=math.floor(math.log10(abs(x))); dec=max(0,2-d)
        return f'{x:.{dec}f}'.replace('.',',')
    plt.scatter([v_kmh],[v_ms],color='red',s=60,zorder=5,label=f'{v_kmh:.0f} km/h = {_tc3(v_ms)} m/s')
    plt.xlabel('velocidad (km/h)'); plt.ylabel('velocidad (m/s)')
    plt.title('Relación km/h ↔ m/s (factor 1/3,6)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(conversion, v_kmh=FloatSlider(value=90.0,min=10,max=150,step=10,description='v (km/h)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    v=90.0*1000/3600
    def _s3(x):
        # formatea a 3 cifras significativas
        if x==0: return "0,00"
        import math
        d=2-int(math.floor(math.log10(abs(x))))
        s=f"{x:.{max(d,0)}f}" if d>=0 else f"{round(x,d):.0f}"
        return s.replace(".",",")
    _pc(f"Velocidad: 90,0 km/h × (1000 m / 1 km) × (1 h / 3600 s) = {v:.3g} m/s")
    _pc(f"Área: 250 cm² × (1,00 m / 100 cm)² = 250 × 1,00×10⁻⁴ = {_s3(250e-4)} m²")
    _pc("Regla práctica: km/h a m/s se divide entre 3,6.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Una velocidad de diseño de 90,0 km/h expresada en el Sistema Internacional (m/s) es:",25,0.03,"m/s","90,0/3,6 = 25,0 m/s.", id_preg="Unidad_p1_1", peso=2.0)
quiz_numerico("Un área de sección de 250 cm² expresada en m² es:",0.025,0.04,"m²","250×10⁻⁴ = 0,0250 m².", id_preg="Unidad_p1_2", peso=2.0)
quiz_opcion_multiple("El análisis dimensional (multiplicar por factores iguales a 1) sirve principalmente para:",
    ["Cambiar el valor real de la medida","Convertir unidades y verificar la coherencia de fórmulas","Aumentar las cifras significativas","Eliminar la incertidumbre"],1,
    "Convierte unidades sin alterar la cantidad física y permite chequear fórmulas.", id_preg="Unidad_p1_3", peso=2.0)

# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: al convertir 90,0 km/h a m/s, un estudiante multiplica por 1000/3600. Otro dice que da igual multiplicar o dividir 'porque son los mismos números'. ¿Quién razona bien?",
    ['El primero: el factor 1000 m/1 km y 1 h/3600 s cancela km y h; así 90,0 km/h = 25,0 m/s',
     'El segundo: 1000/3600 y 3600/1000 contienen los mismos números, así que el resultado es el mismo',
     'El primero, pero porque 1000/3600 es menor que 1, y convertir siempre exige un factor menor que 1',
     'Ninguno: para pasar de km/h a m/s hay que multiplicar por 3,6 y no por 1000/3600'],
    0,
    "El factor se orienta para que las unidades de partida se cancelen: 90,0·1000/3600 = 25,0 m/s. Invertirlo da 324 'unidades' sin sentido. La tercera opción elige bien a la persona con una regla falsa: de m/s a km/h el factor es mayor que 1.",
    id_preg='Uni_p1_concep_1', peso=2.0)

---
## Punto 2 — Cifras significativas en cálculos (Ing. General/Metrología)  ★★  [CL, UCM]
Al operar con datos medidos, el resultado no puede tener más precisión que los datos. Se practican
las reglas con  12,5 × 3,20  y con la suma  12,51 + 3,2.

**Producto/cociente:** el resultado lleva tantas cifras significativas como el dato con MENOS.
**Suma/resta:** el resultado lleva tantos decimales como el dato con MENOS decimales.

> 🔧 **Aplicación en ingeniería:** reportar más cifras de las que la medición justifica es un error
> técnico: sugiere una precisión que no existe. Las cifras significativas comunican honestamente la
> calidad de una medida, algo esencial en informes, planos y especificaciones.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe: *"Reporté un resultado con cierto número de cifras significativas. Actúa como evaluador escéptico: pregúntame si mi cantidad de cifras está justificada por la precisión de los datos de entrada, sin decirme el número correcto."* Recuerda: más cifras no significa más precisión.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7.5,3.8))
    # ejemplo: 12,5 (3cs) × 3,0 (2cs) = 38 (2cs, limitado por el menos preciso)
    datos=[('12,5','3 cifras','#27ae60',3),('3,0','2 cifras','#e74c3c',2)]
    for i,(val,lbl,col,cs) in enumerate(datos):
        y=2-i*0.8
        ax.text(0.5,y,val,fontsize=13,color=col,fontweight='bold')
        ax.text(2,y,lbl,fontsize=10,color=col)
        # barritas que representan las cifras
        for j in range(cs):
            ax.add_patch(plt.Rectangle((3.5+j*0.4,y-0.1),0.3,0.2,color=col,alpha=0.7))
    ax.text(0.3,0.3,'12,5 × 3,0 = 38',fontsize=13,color='#2c3e50',fontweight='bold')
    ax.annotate('resultado: 2 cifras\n(limitado por el dato MENOS preciso)',
                xy=(3,0.35),xytext=(4.3,0.6),fontsize=9,color='#c0392b',
                arrowprops=dict(arrowstyle='->',color='#c0392b'))
    ax.text(4,2.5,'La regla del eslabón más débil:\nel resultado no puede ser más preciso que el peor dato',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(0,7); ax.set_ylim(0,3); ax.axis('off')
    ax.set_title('Cifras significativas: el dato menos preciso manda')
    plt.tight_layout(); plt.show()
esquema()

def cifras(valor=3.14159, n_cifras=3):
    import math
    n_cifras=int(n_cifras)  # el slider entrega float; round necesita int
    if valor==0:
        red=0.0
    else:
        d=n_cifras-int(math.floor(math.log10(abs(valor))))-1
        red=round(valor, int(d))
    fig,ax=plt.subplots(figsize=(7,2.8))
    ax.text(0.5,0.68,f'{valor}',fontsize=22,color='#2c3e50',ha='center',fontweight='bold',transform=ax.transAxes)
    ax.annotate('',xy=(0.5,0.42),xytext=(0.5,0.58),arrowprops=dict(arrowstyle='-|>',color='#7f8c8d',lw=2),transform=ax.transAxes)
    txt=str(red).replace(".",",")
    ax.text(0.5,0.28,f'{txt}',fontsize=22,color='#27ae60',ha='center',fontweight='bold',transform=ax.transAxes)
    ax.text(0.5,0.08,f'redondeado a {n_cifras} cifras significativas',fontsize=11,color='#7f8c8d',ha='center',transform=ax.transAxes)
    ax.axis('off'); ax.set_title('Redondeo a cifras significativas')
    plt.tight_layout(); plt.show()

interact(cifras,
    valor=FloatSlider(value=3.14159,min=0.1,max=100,step=0.001,description='valor',readout_format='.4f'),
    n_cifras=FloatSlider(value=3,min=1,max=5,step=1,description='cifras'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    _pc("12,5 × 3,20: ambos tienen 3 cifras → resultado con 3 cifras = 40,0")
    _pc("12,51 (2 decimales) + 3,2 (1 decimal): manda 1 decimal → 15,7")
    _pc("3,14159 con 3 cifras significativas = 3,14")
    _pc("En producto/cociente cuenta el número de CIFRAS; en suma/resta, los DECIMALES.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_opcion_multiple("El número 3,14159 expresado con tres cifras significativas es:",
    ["3,1","3,14","3,142","3,141"],1,"Tres cifras significativas: 3,14.", id_preg="Unidad_p2_4", peso=2.0)
quiz_opcion_multiple("El producto 12,5 × 3,20 (ambos con tres cifras) debe reportarse como:",
    ["40","40,0","40,00","4,0×10¹"],1,"Con tres cifras significativas: 40,0.", id_preg="Unidad_p2_5", peso=2.0)
quiz_opcion_multiple("En la suma 12,51 + 3,2, el resultado correcto es:",
    ["15,71","15,7","16","15,710"],1,"Manda el dato con menos decimales (3,2): 15,7.", id_preg="Unidad_p2_6", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: una calculadora da 12,5×3,20 = 40,0000. Un estudiante reporta 40,0000 'porque es más preciso'. ¿Correcto o solo convincente?",
    ['Solo convincente: ambos datos tienen 3 cifras significativas, así que el resultado se reporta como 40,0',
     'Correcto: la calculadora no comete errores de redondeo, de modo que 40,0000 es el valor exacto',
     'Solo convincente: debe reportarse 40, porque los ceros a la derecha nunca son significativos',
     'Correcto: escribir más decimales reduce la incertidumbre del producto final'],
    0,
    'El resultado no puede ser más preciso que el dato menos preciso (3 c.s.): 40,0. La tercera opción es una regla falsa: en 40,0 el cero tras la coma sí es significativo y comunica la precisión.',
    id_preg='Uni_p2_concep_2', peso=2.0)

---
## Punto 3 — Interpretación de gráficas lineales (Ing. General/Experimental)  ★★★  [PRC, ULC]
En un experimento se obtiene una relación lineal entre dos variables, con los puntos  (0; 2,00)  y
(4,00; 14,0). Se busca la pendiente y la ecuación de la recta.

**Recta:**  y = m·x + b    **Pendiente:**  m = Δy/Δx    **Intercepto:**  b = valor de y cuando x = 0

> 🔧 **Aplicación en ingeniería:** muchas leyes físicas se verifican linealizando datos y midiendo la
> pendiente (que suele ser una constante física: resistencia, constante elástica, etc.). Interpretar
> la pendiente y el intercepto de una gráfica es una destreza central del trabajo experimental.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega la ecuación de una recta y=mx+b a un asistente: *"En un experimento donde grafiqué una ley física, explícame qué representa FÍSICAMENTE la pendiente y qué el intercepto, y por qué la pendiente suele ser la constante que busco. No me des números."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


def esquema():
    fig,ax=plt.subplots(figsize=(7,4))
    x=np.linspace(0,10,50); y=2.5*x+3
    ax.plot(x,y,color='#2980b9',lw=2.5)
    # triángulo de pendiente
    x1,x2=3,7; y1,y2=2.5*x1+3,2.5*x2+3
    ax.plot([x1,x2],[y1,y1],color='#e67e22',lw=2)
    ax.plot([x2,x2],[y1,y2],color='#27ae60',lw=2)
    ax.text((x1+x2)/2,y1-2,'Δx',ha='center',fontsize=11,color='#e67e22',fontweight='bold')
    ax.text(x2+0.3,(y1+y2)/2,'Δy',fontsize=11,color='#27ae60',fontweight='bold')
    ax.scatter([x1,x2],[y1,y2],s=60,color='#2980b9',zorder=5)
    ax.text(5,25,'pendiente = Δy/Δx\n= significado físico\n(ej: velocidad si es x vs t)',
            ha='center',fontsize=9,color='#2c3e50',
            bbox=dict(boxstyle='round',fc='#fdf6e3',ec='#d9a441'))
    ax.text(0.3,3.5,'intercepto',fontsize=8,color='#7f8c8d')
    ax.set_xlabel('variable independiente (x)'); ax.set_ylabel('variable dependiente (y)')
    ax.set_title('La pendiente de una recta tiene significado físico')
    ax.grid(alpha=0.2,linestyle='--'); ax.set_xlim(0,10)
    plt.tight_layout(); plt.show()
esquema()

def grafica(m=3.0, b=2.0):
    x=np.linspace(0,5,100); y=m*x+b
    plt.figure(figsize=(9,4)); plt.plot(x,y,lw=2,color='navy')
    plt.scatter([0,4],[b,m*4+b],color='red',s=60,zorder=5)
    plt.text(0.2,b+0.5,f'b={b:.1f}',color='#c0392b')
    plt.xlabel('x'); plt.ylabel('y'); plt.title(f'y = {m:.1f}·x + {b:.1f}  (pendiente {m:.1f})')
    plt.grid(alpha=.3); plt.tight_layout(); plt.show()

interact(grafica,
    m=FloatSlider(value=3.0,min=-5,max=5,step=0.5,description='pendiente m'),
    b=FloatSlider(value=2.0,min=-5,max=10,step=1,description='intercepto b'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    _pc("Pendiente: m = Δy/Δx = (14,0 − 2,00)/(4,00 − 0) = 12,0/4,00 = 3,00")
    _pc("Intercepto: b = valor de y en x=0 = 2,00")
    _pc("Ecuación de la recta: y = 3,00·x + 2,00")
    _pc("La pendiente suele ser la constante física de interés en un experimento.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Para una recta que pasa por (0; 2,00) y (4,00; 14,0), la pendiente es:",3.0,0.04,"","m=(14,0−2,00)/4,00=3,00.", id_preg="Unidad_p3_12", peso=3.0)
quiz_numerico("El intercepto b de esa recta (valor de y cuando x=0) es:",2.0,0.05,"","b=2,00 (el valor en x=0).", id_preg="Unidad_p3_13", peso=3.0)
quiz_opcion_multiple("En la verificación experimental de una ley física, la pendiente de la gráfica linealizada suele representar:",
    ["Un error de medición","Una constante física de interés","El intercepto","La incertidumbre"],1,
    "La pendiente suele ser la constante buscada (resistencia, constante elástica, etc.).", id_preg="Unidad_p3_14", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma: 'si la ley es y = k·x, un intercepto claramente distinto de cero en el ajuste delata un error sistemático o un efecto que el modelo no incluye'. ¿Correcto o solo convincente?",
    ['Correcto: si la ley predice y = 0 en x = 0, un intercepto mayor que su incertidumbre revela un offset o un efecto no modelado',
     'Correcto: cualquier intercepto distinto de cero, por pequeño que sea, demuestra que la ley y = k·x es falsa',
     'Solo convincente: el intercepto nunca tiene significado físico; es un artefacto numérico del método de mínimos cuadrados y se ignora',
     'Solo convincente: lo correcto es forzar la recta por el origen, y así el problema del intercepto desaparece'],
    0,
    'El estudiante TIENE razón, con una condición: el intercepto debe ser mayor que su incertidumbre. La segunda opción olvida esa condición (todo ajuste da un intercepto pequeño por azar). Forzar la recta por el origen esconde el sesgo en vez de detectarlo.',
    id_preg='Uni_p3_concep_5', peso=2.0)

## Punto 4 — La conversión correcta como asunto de ingeniería  ★★  [CL, UFT, PRC]
Una conversión de unidades mal hecha no es un error académico menor: en ingeniería puede costar millones o vidas. Estos casos reales lo demuestran:
- **Mars Climate Orbiter (NASA, 1999):** se perdió una sonda de 327 millones de dólares porque un equipo trabajó en unidades imperiales (libra-fuerza) y otro en métricas (newton). La nave se desintegró en la atmósfera de Marte.
- **Vuelo 143 de Air Canada (1983), "Gimli Glider":** un avión se quedó sin combustible a 12 000 m porque se cargó en libras en vez de kilogramos. Planeó sin motores hasta un aterrizaje de emergencia.
- **Obra civil e industrial:** confundir psi con kPa, o pulgadas con centímetros, produce piezas que no ensamblan, tuberías sub-dimensionadas o sobrecostos.

Explora cómo un pequeño factor de conversión mal aplicado escala hasta un error enorme.

> 🔧 **Aplicación en ingeniería:** en Colombia, los proveedores internacionales entregan especificaciones en sistemas mixtos (API en imperial, normas europeas en SI). El ingeniero es el último filtro: una conversión verificada, con unidades explícitas en cada paso, es una barrera de seguridad, no un trámite.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe a un asistente: *"Voy a convertir la presión de prueba de una tubería de 150 psi a kPa. Actúa como evaluador escéptico: pídeme que muestre las unidades en cada paso y que verifique que se cancelen correctamente, pero no me des el resultado."* Recuerda: verificar unidades es una barrera de seguridad.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,ax=plt.subplots(figsize=(8.5,4))
    casos=['Mars Climate\nOrbiter (1999)','Gimli Glider\n(1983)','Pieza mal\nconvertida']
    costos=[327,0,0]  # millones USD (simbólico)
    impactos=['327 M USD\nsonda perdida','avión sin\ncombustible','no ensambla /\nsobrecosto']
    colores=['#c0392b','#e67e22','#e74c3c']
    y=range(len(casos))
    ax.barh(list(y),[3,2,1.2],color=colores,alpha=0.5)
    for i,(caso,imp) in enumerate(zip(casos,impactos)):
        ax.text(0.1,i,caso,va='center',ha='left',fontsize=9,fontweight='bold',color='#2c3e50')
        ax.text([3,2,1.2][i]+0.05,i,imp,va='center',fontsize=8,color=colores[i])
    ax.set_xlim(0,4.2); ax.set_yticks([]); ax.set_xticks([])
    ax.set_title('Errores de conversión de unidades con consecuencias reales')
    ax.text(2,-0.8,'La raíz común: no verificar las unidades en cada paso del cálculo',
            ha='center',fontsize=9,color='#7f8c8d',style='italic')
    plt.tight_layout(); plt.show()
esquema()

def escalar_error(magnitud=1000.0, factor_erroneo=1.0):
    # magnitud correcta en SI vs. con factor mal aplicado
    correcto=magnitud
    # si el factor erróneo != 1, el error se propaga
    con_error=magnitud*factor_erroneo
    error_pct=abs(con_error-correcto)/correcto*100 if correcto!=0 else 0
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(['Valor\ncorrecto (SI)','Valor con factor\nmal aplicado'],[correcto,con_error],
           color=['#27ae60','#c0392b'],alpha=0.75,width=0.5)
    ax.text(0,correcto,f'  {correcto:.0f}',ha='center',va='bottom',fontsize=11,fontweight='bold')
    ax.text(1,con_error,f'  {con_error:.0f}',ha='center',va='bottom',fontsize=11,fontweight='bold')
    ax.set_ylabel('valor de la magnitud')
    ax.set_title(f'Un factor mal aplicado (×{factor_erroneo:.3g}) genera {error_pct:.0f}% de error')
    plt.tight_layout(); plt.show()

interact(escalar_error,
    magnitud=FloatSlider(value=1000.0,min=100,max=5000,step=100,description='magnitud'),
    factor_erroneo=FloatSlider(value=1.0,min=0.3,max=3,step=0.1,description='factor erróneo'));

def solucion():
    print("¿POR QUÉ UNA CONVERSIÓN MAL HECHA ES TAN GRAVE EN INGENIERÍA?")
    print("")
    print("1. El error se PROPAGA: una magnitud mal convertida entra en cálculos")
    print("   posteriores (cargas, presiones, tiempos) y contamina todo el diseño.")
    print("")
    print("2. No se detecta a simple vista: 150 psi y 1034 kPa 'parecen' números")
    print("   distintos y correctos; solo el análisis dimensional revela el error.")
    print("")
    print("3. Casos reales:")
    print("   • Mars Climate Orbiter: libra-fuerza vs newton → sonda perdida (327 M USD).")
    print("   • Gimli Glider: combustible en libras vs kilogramos → avión sin combustible.")
    print("")
    print("BUENAS PRÁCTICAS:")
    print("   • Escribir las unidades en CADA paso (análisis dimensional).")
    print("   • Verificar que las unidades se cancelen correctamente.")
    print("   • Confirmar el factor de conversión antes de usarlo.")
    print("   • Reportar siempre con unidades explícitas en el SI.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_opcion_multiple("¿Cuál fue la causa raíz de la pérdida del Mars Climate Orbiter (1999)?",
    ["Una falla del motor","Un equipo usó unidades imperiales (libra-fuerza) y otro métricas (newton), sin conversión","Mal clima en Marte","Error de programación no relacionado con unidades"],1,
    "Dos equipos trabajaron en sistemas de unidades distintos sin convertir: la mezcla imperial/métrica desintegró la sonda. Es el caso emblemático de por qué la conversión importa.", id_preg="Uni_p4_q_1", peso=2.0)
quiz_opcion_multiple("¿Por qué un error de conversión es especialmente peligroso en un cálculo de ingeniería?",
    ["Porque se ve fácilmente","Porque se PROPAGA a los cálculos posteriores y no se detecta a simple vista","Porque solo afecta la estética","Porque las unidades no importan al final"],1,
    "Una magnitud mal convertida entra en cálculos siguientes y contamina todo el diseño, sin ser evidente. Por eso el análisis dimensional en cada paso es una barrera de seguridad.", id_preg="Uni_p4_q_2", peso=2.0)
quiz_opcion_multiple("¿Cuál es la mejor práctica para evitar errores de conversión?",
    ["Confiar en la memoria","Escribir las unidades en cada paso y verificar que se cancelen (análisis dimensional)","Usar solo números sin unidades","Redondear agresivamente"],1,
    "El análisis dimensional —arrastrar las unidades en cada paso y verificar que se cancelen— es la herramienta más efectiva para atrapar errores de conversión antes de que se propaguen.", id_preg="Uni_p4_q_3", peso=2.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice 'convertí rápido de memoria, el número parece razonable, así que está bien'. ¿Correcto o solo convincente?",
    ['Solo convincente: que parezca razonable no verifica nada; hay que seguir las unidades paso a paso hasta que se cancelen',
     'Correcto: si el orden de magnitud es razonable, cualquier error que quede es despreciable en ingeniería',
     'Correcto: los errores de conversión siempre producen resultados absurdos que se notan a simple vista',
     'Solo convincente: convertir de memoria está bien; el único riesgo es reportar demasiadas cifras significativas'],
    0,
    "Un error de factor 4,45 (lbf↔N) o 2,2 (lb↔kg) deja números 'razonables': así se perdió el Mars Climate Orbiter y el Gimli Glider se quedó sin combustible. Solo el análisis dimensional explícito verifica la conversión.",
    id_preg='Uni_p4_concep_4', peso=2.0)

## Punto 5 — El reporte adecuado en informes de ingeniería  ★★  [ULC, PRC, UCM]
Un resultado de ingeniería mal reportado es tan problemático como uno mal calculado: quien lee el informe toma decisiones a partir de él. Un buen reporte comunica no solo el valor, sino también su **confiabilidad**. Reglas de oro:
- **Unidades siempre explícitas:** un número sin unidad es ambiguo y peligroso.
- **Cifras significativas coherentes:** no reportar más precisión de la que los datos justifican.
- **Incertidumbre bien expresada:** valor ± incertidumbre, con decimales coherentes (ej. 10,3 ± 0,2 cm, no 10,30 ± 0,16).
- **Notación consistente:** en Colombia, coma decimal; separador de miles claro.
- **Trazabilidad:** indicar el instrumento, su resolución y las condiciones de medida.

Explora cómo un mismo resultado se ve profesional o descuidado según cómo se reporte.

> 🔧 **Aplicación en ingeniería:** un informe de integridad de tuberías (API/ASME) o un acta topográfica que no reporta unidades e incertidumbres correctamente puede ser rechazado en auditoría, o peor, llevar a decisiones equivocadas. El reporte es el producto final del ingeniero: su claridad y rigor lo definen profesionalmente.

> 🤖 **TIP-IA — Abogado del diablo:**
> Pídele a un asistente: *"Aquí está mi resultado reportado: [tu resultado]. Actúa como un auditor exigente y critica mi reporte: ¿faltan unidades?, ¿las cifras significativas son coherentes?, ¿la incertidumbre está bien redondeada?, ¿la notación es consistente? No me lo corrijas, solo señala las debilidades."*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))

def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,ax=plt.subplots(figsize=(9,4))
    ax.axis('off')
    # ejemplo mal vs bien reportado
    ax.text(0.25,0.92,'REPORTE DEFICIENTE  [X]',ha='center',fontsize=12,fontweight='bold',color='#c0392b',transform=ax.transAxes)
    malos=['L = 10.304','A = 25,3958 cm2','P = 1034.5678','sin instrumento','v = 25']
    for i,t in enumerate(malos):
        ax.text(0.25,0.80-i*0.13,t,ha='center',fontsize=10,color='#c0392b',family='monospace',transform=ax.transAxes)
    ax.text(0.75,0.92,'REPORTE PROFESIONAL  [OK]',ha='center',fontsize=12,fontweight='bold',color='#27ae60',transform=ax.transAxes)
    buenos=['L = (10,3 ± 0,2) cm','A = (25,0 ± 0,5) cm²','P = 1,03 MPa','regla, res. 1 mm','v = 25,0 m/s']
    for i,t in enumerate(buenos):
        ax.text(0.75,0.80-i*0.13,t,ha='center',fontsize=10,color='#27ae60',family='monospace',transform=ax.transAxes)
    # línea divisoria
    ax.axvline(0.5,ymin=0.1,ymax=0.9,color='#bdc3c7',lw=1.5,ls='--')
    ax.text(0.5,0.02,'El mismo resultado: uno inspira confianza, el otro genera dudas',
            ha='center',fontsize=9,color='#7f8c8d',style='italic',transform=ax.transAxes)
    ax.set_title('Cómo se reporta un resultado en ingeniería')
    plt.tight_layout(); plt.show()
esquema()

def reporte(valor=10.34, incertidumbre=0.16):
    import math
    # redondeo correcto: incertidumbre a 1 cifra sig, valor al mismo decimal
    if incertidumbre>0:
        d=math.floor(math.log10(incertidumbre)); dec=-d
        inc_red=round(incertidumbre,dec)
        val_red=round(valor,dec)
    else:
        inc_red=0; val_red=valor; dec=1
    fig,ax=plt.subplots(figsize=(8,3.2))
    ax.axis('off')
    ax.text(0.5,0.75,'Entrada cruda:',ha='center',fontsize=10,color='#7f8c8d',transform=ax.transAxes)
    ax.text(0.5,0.58,f'valor={valor}, incertidumbre={incertidumbre}',ha='center',fontsize=11,family='monospace',color='#c0392b',transform=ax.transAxes)
    ax.text(0.5,0.32,'Reporte correcto:',ha='center',fontsize=10,color='#7f8c8d',transform=ax.transAxes)
    txt=f'({val_red:.{max(dec,0)}f} ± {inc_red:.{max(dec,0)}f}) cm'.replace('.',',')
    ax.text(0.5,0.13,txt,ha='center',fontsize=15,fontweight='bold',color='#27ae60',transform=ax.transAxes)
    ax.set_title('Redondeo: incertidumbre a 1 cifra significativa, valor al mismo decimal')
    plt.tight_layout(); plt.show()

interact(reporte,
    valor=FloatSlider(value=10.34,min=5,max=20,step=0.01,description='valor'),
    incertidumbre=FloatSlider(value=0.16,min=0.01,max=1.0,step=0.01,description='incertid.'));

def solucion():
    print("ANATOMÍA DE UN BUEN REPORTE:")
    print("")
    print("1. UNIDADES explícitas: 'v = 25' es ambiguo; 'v = 25,0 m/s' es claro.")
    print("")
    print("2. CIFRAS SIGNIFICATIVAS coherentes: no escribir 25,3958 cm² si los datos")
    print("   solo justifican 3 cifras → 25,4 cm².")
    print("")
    print("3. INCERTIDUMBRE bien redondeada: a UNA cifra significativa, y el valor")
    print("   con los mismos decimales:")
    print("   ❌ 10,304 ± 0,16      ✅ (10,3 ± 0,2) cm")
    print("")
    print("4. NOTACIÓN consistente: coma decimal (Colombia), unidades del SI.")
    print("")
    print("5. TRAZABILIDAD: indicar instrumento y resolución (ej. 'regla, res. 1 mm').")
    print("")
    print("El reporte es el producto final del ingeniero: comunica el resultado Y su confiabilidad.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 5 ---")
quiz_opcion_multiple("¿Cuál de estos reportes está correctamente expresado?",
    ["L = 10.304 ± 0.16","L = (10,3 ± 0,2) cm","L = 10,304","L = 10 ± 0,163 cm"],1,
    "La incertidumbre va a 1 cifra significativa (0,2) y el valor con los mismos decimales (10,3), con unidad explícita y coma decimal: (10,3 ± 0,2) cm.", id_preg="Uni_p5_q_5", peso=2.0)
quiz_opcion_multiple("¿Por qué reportar 'v = 25' (sin unidad) es un problema en un informe?",
    ["No es problema","Es ambiguo y peligroso: ¿25 m/s, km/h, nudos? Un número sin unidad no comunica una magnitud física","Ocupa menos espacio","Es más elegante"],1,
    "Un número sin unidad es ambiguo: 25 m/s y 25 km/h son muy distintos. En ingeniería, omitir la unidad puede llevar a decisiones equivocadas.", id_preg="Uni_p5_q_6", peso=2.0)
quiz_opcion_multiple("En un informe, ¿por qué se indica el instrumento y su resolución?",
    ["Por formalidad","Por trazabilidad: permite saber la confiabilidad de la medida y reproducirla","No es necesario","Para hacerlo más largo"],1,
    "Indicar el instrumento y su resolución da trazabilidad: el lector sabe qué tan confiable es el dato y puede reproducir la medición. Es parte del rigor profesional.", id_preg="Uni_p5_q_7", peso=2.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante reporta 'A = 25,3958 cm²' a partir de lados medidos con regla, y dice que es mejor porque tiene más decimales. ¿Correcto o solo convincente?",
    ['Solo convincente: los datos solo justifican 3 cifras: A = 25,4 cm²',
     'Correcto: cada decimal adicional reduce la incertidumbre del área calculada',
     'Solo convincente: en un informe técnico un área siempre se reporta con 2 cifras significativas, es decir 25 cm²',
     'Correcto: si la calculadora muestra los dígitos, todos ellos son significativos'],
    0,
    "La precisión del reporte la fijan los datos, no la calculadora. Tampoco existe una regla de '2 cifras para áreas': se usan las que justifican las medidas (aquí 3).",
    id_preg='Uni_p5_concep_8', peso=2.0)

---
## ✅ Fin del taller de Unidades, Medidas e Incertidumbre (Física BR)
Estas bases —unidades, cifras significativas, incertidumbre y gráficas— se aplican en TODOS los
demás talleres. Experimenta con los sliders y **predice** antes de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=20)
